# **Used Car Price Prediction**

### **Imports**

In [4]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from datetime import datetime

### **Custom transformers**

In [2]:
class DataCleaner(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if 'price' in X.columns:
            X['price'] = X['price'].str.replace("$", "").str.replace(",","").astype(float)
        
        X['milage'] = X['milage'].str.replace('mi.',"").str.replace(" mi.", "").str.replace(",", "").astype(float)
        
        X['accident'] = X['accident'].fillna('Unknown')
        X['had_accident'] = X['accident'].apply(lambda x: 1 if 'accident' in x.lower() else 0)
        X['accident_unknown'] = X['accident'].apply(lambda x: 1 if 'Unknown' else 0)
        X = X.drop(columns=['accident'])

        X['clean_title'] = X['clean_title'].fillna("No")
        X['clean_title'] = X['clean_title'].map({'Yes':1, 'No':1})

        X['fuel_type'] = X['fuel_type'].fillna('Unknown')

        return X

In [5]:
class FeatureEngineering(BaseEstimator, TransformerMixin):

    def __init__(self, current_year=None, luxury_brands=None):
        self.current_year = current_year if current_year else datetime.datetime.now().year
        self.luxury_brands = luxury_brands if luxury_brands else [
            'Lexus', 'INFINITI', 'Audi', 'Acura', 'BMW', 'Tesla', 'Land', 
            'Aston', 'Lincoln', 'Jaguar', 'Mercedes-Benz', 'Genesis', 'Bentley', 
            'Lucid', 'Porsche', 'Cadillac', 'Lamborghini', 
            'Maserati', 'Rivian', 'Alfa', 'Ferrari', 'Bugatti', 
            'Rolls-Royce', 'McLaren', 'Lotus', 'Karma', 'Maybach'
        ]
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X['car_age'] = self.current_year - X['model_year']

        X['horsepower'] = X['engine'].str.extract(r"(\d+\.?\d*)HP", expand=False).astype(float)
        X['horsepower_missing'] = X['horsepower'].isna().astyp
        
        X['milage_per_year'] = X['milage'] / (X['car_age'] + 1)

        X['is_luxury'] = X['brand'].apply(lambda x: 1 if x in self.luxury_brands else 0)

        fuel_dummies = pd.get_dummies(X['fuel_type'], prefix='fuel', drop_first=True).astype(int)
        X = pd.concat([X, fuel_dummies], axis=1)
        X = X.drop(columns=['fuel_type'])

        return X

In [7]:
class TargetEncoder(BaseEstimator, TransformerMixin):

    def __init__(self, column='brand', n_splits=5):
        self.column = column
        self.n_splits = n_splits
        self.encoding_map_ = None
        self.global_mean_ = None
    
    def fit(self, X, y=None):
        if y is None:
            raise ValueError("y is required for target encoding")
        
        X = X.copy()
        self.global_mean_ = y.mean()
        self.encoding_map_ = pd.DataFrame({self.column: X[self.column], 'target':y}).groupby(self.column)['target'].mean().to_dict()

        return self
    
    def transform(self, X):
        X = X.copy()

        X[f'{self.column}_encoded'] = X[self.column].map(self.encoding_map_).fillna(self.global_mean_)
        X = X.drop(columns=[self.column])
    
        return X

In [8]:
class OutlierRemover(BaseEstimator, TransformerMixin):

    def __init__(self, column='price', multiplier=1.5):
        self.column = column
        self.multiplier = multiplier
        self.lower_bound_ = None
        self.upper_bound_ = None
    
    def fit(self, X, y=None):
        Q1 = X[self.column].quantile(0.25)
        Q3 = X[self.column].quantile(0.75)
        IQR = Q3 - Q1

        self.lower_bound_ = Q1 - self.multiplier * IQR
        self.upper_bound_ = Q3 + self.multiplier * IQR
    
    def transform(self, X):
        X = X.copy()
        mask = (X[self.column] >= self.lower_bound_) & (X[self.column] <= self.upper_bound_)
        removed = len(X) - mask.sum()
        print(f"OutlierRemover: removed {removed:,} outliers ({removed/len(X)*100:.1f}%)")
        
        return X[mask]

In [9]:
class ColumnSelector(BaseEstimator, TransformerMixin):

    def __init__(self, columns):
        self.columns = columns
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return X[self.columns].copy()